# 02 — Inferência: MeshAnything V2 + Baselines

Este notebook:
1. Instala dependências e clona os repos (MeshAnythingV2, MeshAnything)
2. Carrega point clouds de `data/pointclouds/`
3. Roda inferência com **MeshAnything V2** (modelo principal)
4. Roda inferência com **MeshAnything V1** (baseline naive)
5. Salva cada mesh na pasta correspondente em `data/meshes/`

In [ ]:
# Cell 1 — Instalar dependências
!pip install -q -r requirements.txt
!pip install -q flash-attn --no-build-isolation

In [ ]:
# Cell 2 — Clonar repos (pula se já existem)
import os

repos = {
    "MeshAnythingV2": "https://github.com/buaacyw/MeshAnythingV2.git",
    "MeshAnything": "https://github.com/buaacyw/MeshAnything.git",
}

for name, url in repos.items():
    if not os.path.exists(name):
        !git clone {url}
    else:
        print(f"{name} já existe, pulando clone.")

In [ ]:
# Cell 3 — Setup de paths
PROJECT_ROOT = os.path.abspath(".")
DATA_PC = os.path.join(PROJECT_ROOT, "data", "pointclouds")
MESH_MAV2 = os.path.join(PROJECT_ROOT, "data", "meshes", "meshanything_v2")
MESH_NAIVE = os.path.join(PROJECT_ROOT, "data", "meshes", "baselines", "naive")

os.makedirs(MESH_MAV2, exist_ok=True)
os.makedirs(MESH_NAIVE, exist_ok=True)

# Listar point clouds disponíveis (.npy)
pc_files = sorted([
    f for f in os.listdir(DATA_PC)
    if f.endswith(".npy")
])
print(f"Point clouds encontradas: {len(pc_files)}")
for f in pc_files:
    print(f"  - {f}")

## MeshAnything V2 (Modelo Principal)

Usa AMT (Adjacent Mesh Tokenization) — gera meshes com até 1600 faces.

In [ ]:
# Cell 4 — Rodar MeshAnything V2
import subprocess
import shutil

MAV2_DIR = os.path.join(PROJECT_ROOT, "MeshAnythingV2")

# Criar symlink das point clouds no diretório de input do V2
v2_input = os.path.join(MAV2_DIR, "input_pc")
if os.path.exists(v2_input):
    shutil.rmtree(v2_input)
os.symlink(DATA_PC, v2_input)

print("Rodando MeshAnything V2...")
result = subprocess.run(
    [
        "python", "main.py",
        "--input_dir", "input_pc",
        "--out_dir", "output_v2",
        "--input_type", "pc_normal",
    ],
    cwd=MAV2_DIR,
    capture_output=True,
    text=True,
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("ERRO:", result.stderr[-2000:])
else:
    print("MeshAnything V2 concluído com sucesso.")

In [ ]:
# Cell 5 — Mover meshes geradas pelo V2 para data/meshes/meshanything_v2/
import glob

v2_output_dir = os.path.join(MAV2_DIR, "output_v2")
v2_meshes = glob.glob(os.path.join(v2_output_dir, "**", "*.obj"), recursive=True)

print(f"Meshes geradas pelo V2: {len(v2_meshes)}")
for mesh_path in v2_meshes:
    mesh_name = os.path.basename(mesh_path)
    dest = os.path.join(MESH_MAV2, mesh_name)
    shutil.copy2(mesh_path, dest)
    print(f"  {mesh_name} → {MESH_MAV2}")

## Baseline Naive (MeshAnything V1)

Tokenização naive sem AMT — cada face usa 3 vértices independentes. Gera meshes com até 800 faces.

In [ ]:
# Cell 6 — Rodar MeshAnything V1 (baseline naive)
MAV1_DIR = os.path.join(PROJECT_ROOT, "MeshAnything")

# Criar symlink das point clouds no diretório de input do V1
v1_input = os.path.join(MAV1_DIR, "input_pc")
if os.path.exists(v1_input):
    shutil.rmtree(v1_input)
os.symlink(DATA_PC, v1_input)

print("Rodando MeshAnything V1 (baseline naive)...")
result = subprocess.run(
    [
        "python", "main.py",
        "--input_dir", "input_pc",
        "--out_dir", "output_v1",
        "--input_type", "pc_normal",
    ],
    cwd=MAV1_DIR,
    capture_output=True,
    text=True,
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("ERRO:", result.stderr[-2000:])
else:
    print("MeshAnything V1 concluído com sucesso.")

In [ ]:
# Cell 7 — Mover meshes geradas pelo V1 para data/meshes/baselines/naive/
v1_output_dir = os.path.join(MAV1_DIR, "output_v1")
v1_meshes = glob.glob(os.path.join(v1_output_dir, "**", "*.obj"), recursive=True)

print(f"Meshes geradas pelo V1 (naive): {len(v1_meshes)}")
for mesh_path in v1_meshes:
    mesh_name = os.path.basename(mesh_path)
    dest = os.path.join(MESH_NAIVE, mesh_name)
    shutil.copy2(mesh_path, dest)
    print(f"  {mesh_name} → {MESH_NAIVE}")

In [ ]:
# Cell 7 — Resumo
import glob

print("=" * 60)
print("RESUMO — Passo 2: Inferência")
print("=" * 60)

for label, path in [
    ("Ground truth (TripoSR)", os.path.join(PROJECT_ROOT, "data", "meshes", "ground_truth")),
    ("MeshAnything V2", MESH_MAV2),
    ("Baseline naive (V1)", MESH_NAIVE),
]:
    meshes = glob.glob(os.path.join(path, "*.obj"))
    print(f"\n{label}: {len(meshes)} meshes")
    for m in meshes:
        print(f"  - {os.path.basename(m)}")

print(f"\nPróximo passo: 03_evaluation.ipynb (métricas + tabelas)")